https://github.com/NielsRogge/Transformers-Tutorials/blob/master/Mistral/Supervised_fine_tuning_(SFT)_of_an_LLM_using_Hugging_Face_tooling.ipynb

In [16]:
from huggingface_hub import login
login()

In [8]:
from datasets import load_dataset

# based on config
train_raw_datasets = load_dataset("HuggingFaceH4/ultrachat_200k", split='train_sft')
test_raw_datasets = load_dataset("HuggingFaceH4/ultrachat_200k", split='test_sft')

The dataset contains various splits, each with a certain number of rows. In our case, as we're going to do supervised fine-tuning (SFT), only the "train_sft" and "test_sft" splits are relevant for us.

In [9]:
from datasets import DatasetDict

# remove this when done debugging
indices = range(0,100)

dataset_dict = {"train": train_raw_datasets.select(indices),
                "test": test_raw_datasets.select(indices)}

raw_datasets = DatasetDict(dataset_dict)
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 100
    })
    test: Dataset({
        features: ['prompt', 'prompt_id', 'messages'],
        num_rows: 100
    })
})

In [10]:
example = raw_datasets["train"][0]
print(example.keys())

dict_keys(['prompt', 'prompt_id', 'messages'])


Each message is a dictionary containing 2 keys, namely:

- "role": specifies who the creator of the message is (could be "system", "assistant" or "user" - the latter referring to a human).
- "content": the actual content of the message.

Let's print out the sequence of messages for this training example:

In [12]:
messages = example["messages"]
for message in messages:
  role = message["role"]
  content = message["content"]
  print('{0:20}:  {1}'.format(role, content))

user                :  These instructions apply to section-based themes (Responsive 6.0+, Retina 4.0+, Parallax 3.0+ Turbo 2.0+, Mobilia 5.0+). What theme version am I using?
On your Collections pages & Featured Collections sections, you can easily show the secondary image of a product on hover by enabling one of the theme's built-in settings!
Your Collection pages & Featured Collections sections will now display the secondary product image just by hovering over that product image thumbnail.
Does this feature apply to all sections of the theme or just specific ones as listed in the text material?
assistant           :  This feature only applies to Collection pages and Featured Collections sections of the section-based themes listed in the text material.
user                :  Can you guide me through the process of enabling the secondary image hover feature on my Collection pages and Featured Collections sections?
assistant           :  Sure, here are the steps to enable the secondary 

В этом случае, похоже, инструкции касаются включения определенных функций в Shopify. Интересно!

### Load tokenizer

Далее мы создаем экземпляр токенизатора.

Мы также устанавливаем некоторые атрибуты, которые токенизатор базовой модели обычно не устанавливает, например:

- **padding token ID**. Во время pre-training padding не нужен, так как вы просто создаете блоки текста для прогнозирования следующего токена, но во время тонкой настройки нам нужно будет заполнить пары (инструкция, завершение), чтобы создать батчи одинаковой длины.
- **model max length**: это необходимо для того, чтобы обрезать последовательности, которые слишком длинные для модели. Здесь мы решили обучаться не более чем на 2048 токенах.
- **chat template**. Шаблон чата определяет, как каждый список сообщений преобразуется в токенизированную строку, добавляя специальные строки между ними, такие как <|user|> для обозначения сообщения пользователя и <|assistant|> для обозначения ответа чат-бота. Здесь мы определяем шаблон чата по умолчанию, используемый большинством моделей чата.

In [17]:
from transformers import AutoTokenizer

model_id = "mistralai/Mistral-7B-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# set pad_token_id equal to the eos_token_id if not set
if tokenizer.pad_token_id is None:
  tokenizer.pad_token_id = tokenizer.eos_token_id

# Set reasonable default for models without max length
if tokenizer.model_max_length > 100_000:
  tokenizer.model_max_length = 2048

# Set chat template
DEFAULT_CHAT_TEMPLATE = "{% for message in messages %}\n{% if message['role'] == 'user' %}\n{{ '<|user|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'system' %}\n{{ '<|system|>\n' + message['content'] + eos_token }}\n{% elif message['role'] == 'assistant' %}\n{{ '<|assistant|>\n'  + message['content'] + eos_token }}\n{% endif %}\n{% if loop.last and add_generation_prompt %}\n{{ '<|assistant|>' }}\n{% endif %}\n{% endfor %}"
tokenizer.chat_template = DEFAULT_CHAT_TEMPLATE

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

### Apply chat template

После того, как мы снабдили токенизатор соответствующими атрибутами, пришло время применить шаблон чата к каждому списку сообщений. Здесь мы в основном превращаем каждый список сообщений (инструкции, завершения) в токенизированную строку для модели.

Обратите внимание, что мы указываем здесь tokenize=False, поскольку SFTTrainer, который мы определим позже, выполнит токенизацию внутри. Здесь мы только превращаем список сообщений в строки с тем же форматом.

In [18]:
# todo num_proc=cpu_count(),
# todo fn_kwargs={"tokenizer": tokenizer},

import re
import random
from multiprocessing import cpu_count

def apply_chat_template(example, tokenizer):
    messages = example["messages"]
    # добавляем пустое system message если первая роль не system
    # system используется для указания как себя вести, если из данных это не известно, 
    # то мы не можем задавать какой-то контент
    if messages[0]["role"] != "system":
        messages.insert(0, {"role": "system", "content": ""})
    example["text"] = tokenizer.apply_chat_template(messages, tokenize=False)

    return example

column_names = list(raw_datasets["train"].features)
raw_datasets = raw_datasets.map(apply_chat_template,
                                num_proc=cpu_count(),
                                fn_kwargs={"tokenizer": tokenizer},
                                remove_columns=column_names,
                                desc="Applying chat template",)

# create the splits
train_dataset = raw_datasets["train"]
eval_dataset = raw_datasets["test"]

for index in random.sample(range(len(raw_datasets["train"])), 3):
  print(f"Sample {index} of the processed training set:\n\n{raw_datasets['train'][index]['text']}")

num_proc must be <= 100. Reducing num_proc to 100 for dataset of size 100.


Applying chat template (num_proc=100):   0%|          | 0/100 [00:00<?, ? examples/s]

num_proc must be <= 100. Reducing num_proc to 100 for dataset of size 100.


Applying chat template (num_proc=100):   0%|          | 0/100 [00:00<?, ? examples/s]

Sample 60 of the processed training set:

<|system|>
</s>
<|user|>
What is the traditional dress worn for a wedding in Greece?</s>
<|assistant|>
The traditional dress worn for a wedding in Greece is called a "Fustanella," which is a traditional Greek skirt and a white shirt, usually with a vest or jacket. This is worn by the groom and his male family members, while the bride and female family members wear a white dress or gown. The traditional dress is often complemented with accessories such as a headpiece, jewelry, and shoes.</s>
<|user|>
Can you tell me more about the traditional Greek wedding customs besides the dress?</s>
<|assistant|>
Sure, here are some traditional Greek wedding customs:

1. Engagement: Before a Greek wedding, the couple typically gets engaged, also known as "kroiazi." It is a formal announcement of their intention to get married in front of their family and friends.

2. Pre-wedding customs: A day or two before the wedding, the couple often hosts a pre-wedding c

In [19]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 100
    })
    test: Dataset({
        features: ['text'],
        num_rows: 100
    })
})

### Define model arguments

Далее пришло время определить аргументы модели.

Здесь требуется некоторое пояснение относительно способов тонкой настройки модели.

#### Full fine-tuning

Обычно выполняется «полная тонкая настройка»: это означает, что мы просто обновим все веса базовой модели во время тонкой настройки. Затем это обычно делается либо с полной точностью (float32), либо со смешанной точностью (mixed precision) (комбинация float32 и float16). Однако с более крупными моделями, такими как LLM, это становится невозможным.

Для справки, float32 означает, что каждый параметр модели сохраняется в 32 битах или 4 байтах. Следовательно, для модели с 7 миллиардами параметров, такой как Mistral-7B, требуется 7 миллиардов параметров * 4 байта на параметр = 28 ГБ оперативной памяти графического процессора, только для загрузки модели. Во время обучения с оптимизатором вроде AdamW требуется память не только для модели, но и для градиентов и состояний оптимизатора, что примерно в 18 раз превышает размер модели в гигабайтах при обучении со смешанной точностью, в этом случае 7 * 18 = 126 ГБ оперативной памяти GPU. И это только для модели с 7B параметрами! Более подробную информацию см. в руководстве: https://huggingface.co/docs/transformers/v4.20.1/en/perf_train_gpu_one.


#### LoRa, a PEFT method

Поэтому некоторые умные люди в Microsoft придумали метод под названием LoRa (адаптация низкого ранга). Идея здесь заключается в том, что вместо выполнения полной тонкой настройки мы собираемся заморозить существующую модель и только добавить несколько весовых коэффициентов параметров к модели (называемых «адаптерами»), которые мы собираемся обучить.

LoRa — это то, что мы называем методом тонкой настройки с эффективными параметрами (PEFT). Это популярный метод тонкой настройки моделей эффективным с точки зрения параметров способом, обучая только несколько адаптеров, оставляя существующую модель нетронутой. LoRa доступен в библиотеке PEFT от Hugging Face, которая также поддерживает различные другие методы PEFT (но LoRa является самым популярным на момент написания статьи).


#### QLoRa, an even more efficient method

С обычным LoRa можно было бы хранить базовую модель в 32 или 16 битах в памяти, а затем обучать веса параметров. Однако были разработаны новые методы, позволяющие значительно уменьшить размер модели, до 8 или 4 бит на параметр (мы называем это «квантованием»). Следовательно, если мы применяем LoRa к квантованной модели (например, 4-битной модели), то мы называем это QLoRa. У нас есть запись в блоге, в которой рассказывается обо всем этом. Существуют различные методы квантования, здесь мы собираемся использовать интеграцию BitsandBytes.

In [24]:
from transformers import BitsAndBytesConfig
import torch

# specify how to quantize the model
quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype="bfloat16",
)
device_map = {"": torch.cuda.current_device()} if torch.cuda.is_available() else None

model_kwargs = dict(
    attn_implementation="flash_attention_2", # set this to True if your GPU supports it (Flash Attention drastically speeds up model computations)
    torch_dtype="auto",
    use_cache=False, # set to False as we're going to use gradient checkpointing
    device_map=device_map,
    quantization_config=quantization_config,
)

### Define SFTTrainer

Далее мы определяем SFTTrainer, доступный в библиотеке TRL. Этот класс наследует от класса Trainer, доступного в библиотеке Transformers, но специально оптимизирован для supervised fine-tuning (instruction tuning). Его можно использовать для обучения из коробки на одном или нескольких графических процессорах, используя Accelerate в качестве бэкэнда.

В частности, он поддерживает packing dataset (https://huggingface.co/docs/trl/sft_trainer#packing-dataset--constantlengthdataset-), когда несколько коротких примеров упаковываются в одну и ту же входную последовательность для повышения эффективности обучения.

Поскольку мы собираемся использовать QLoRa, библиотека PEFT предоставляет удобный LoraConfig, который определяет, на каких слоях базовой модели применять адаптеры. Обычно LoRa применяется к матрицам линейной проекции слоев внимания Transformer. Затем мы предоставляем эту конфигурацию классу SFTTrainer. Веса базовой модели будут загружены, когда мы укажем model_id (это требует некоторого времени).

Мы также указываем различные гиперпараметры, касающиеся обучения, такие как:
- fine-tune для 1 эпохи
- learning rate и scheduler
- мы собираемся использовать gradient checkpointing (еще один способ экономии памяти во время обучения)
- и так далее.

In [ ]:
from trl import SFTTrainer
from peft import LoraConfig
from transformers import TrainingArguments

# path where the Trainer will save its checkpoints and logs
output_dir = 'data/zephyr-7b-sft-lora'

# based on config
training_args = TrainingArguments(
    fp16=True, # specify bf16=True instead when training on GPUs that support bf16
    do_eval=True,
    evaluation_strategy="epoch",
    gradient_accumulation_steps=128,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=2.0e-05,
    log_level="info",
    logging_steps=5,
    logging_strategy="steps",
    lr_scheduler_type="cosine",
    max_steps=-1,
    num_train_epochs=1,
    output_dir=output_dir,
    overwrite_output_dir=True,
    per_device_eval_batch_size=1, # originally set to 8
    per_device_train_batch_size=1, # originally set to 8
    # push_to_hub=True,
    # hub_model_id="zephyr-7b-sft-lora",
    # hub_strategy="every_save",
    # report_to="tensorboard",
    save_strategy="no",
    save_total_limit=None,
    seed=42,
)

# based on config
peft_config = LoraConfig(
        r=64,
        lora_alpha=16,
        lora_dropout=0.1,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

trainer = SFTTrainer(
        model=model_id,
        model_init_kwargs=model_kwargs,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        dataset_text_field="text",
        tokenizer=tokenizer,
        packing=True,
        peft_config=peft_config,
        max_seq_length=tokenizer.model_max_length,
    )

In [ ]:

train_result = trainer.train()

In [ ]:
metrics = train_result.metrics
max_train_samples = training_args.max_train_samples if training_args.max_train_samples is not None else len(train_dataset)
metrics["train_samples"] = min(max_train_samples, len(train_dataset))
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)
trainer.save_state()

### Inference

Давайте сгенерируем несколько новых текстов с помощью нашей обученной модели.

Для инференса есть 2 основных способа:

- использование pipeline(), который абстрагирует для нас множество деталей, касающихся предварительной и постобработки. Например, эта карта модели иллюстрирует это.
- использование классов AutoTokenizer и AutoModelForCausalLM самостоятельно и реализация деталей самостоятельно.
Давайте сделаем последнее, чтобы понять, что происходит.

Начнем с загрузки модели из каталога, в котором мы сохранили веса. Мы также указываем использование 4-битного инференса и автоматическое размещение модели на доступных графических процессорах (см. документацию относительно device_map="auto" - https://huggingface.co/docs/accelerate/concept_guides/big_model_inference#the-devicemap).


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(output_dir)
model = AutoModelForCausalLM.from_pretrained(output_dir, load_in_4bit=True, device_map="auto")

Далее мы готовим список сообщений для модели с помощью шаблона чата токенизатора. Обратите внимание, что мы также добавляем сюда сообщение «system», чтобы указать модели, как себя вести. Во время обучения мы добавляли пустое системное сообщение к каждому разговору.

Мы также указываем add_generation_prompt=True, чтобы убедиться, что модель получает запрос на генерацию ответа (это полезно во время инференса). 
Мы указываем «cuda», чтобы переместить входы в графический процессор. Модель будет автоматически на графическом процессоре, поскольку мы использовали device_map="auto" выше.

Далее мы используем метод generate() для авторегрессивной генерации следующих идентификаторов токенов, одного за другим. Обратите внимание, что существуют различные стратегии генерации, такие как жадное декодирование или поиск луча. Здесь мы используем sampling.

Наконец, мы используем метод batch_decode токенизатора, чтобы превратить сгенерированные идентификаторы токенов обратно в строки.

In [ ]:
import torch

# We use the tokenizer's chat template to format each message - see https://huggingface.co/docs/transformers/main/en/chat_templating
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
]

# prepare the messages for the model
input_ids = tokenizer.apply_chat_template(messages, truncation=True, add_generation_prompt=True, return_tensors="pt").to("cuda")

# inference
outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95
)
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])